# Batch Query Test Against Known-Pose Map

This notebook localizes test/query images against the known-pose reconstruction produced by `reconstruction_known_pose.ipynb`. Query images can come from any camera as long as their basenames exist in `metadata/poses.json`.

The map and query cameras do not need to share the same camera_name. Ground truth is matched by image basename.

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime
import copy
import json
import random
import shutil

import numpy as np
import pandas as pd
import pycolmap
import torch

from hloc import extract_features, match_features, pairs_from_retrieval
from hloc.localize_sfm import QueryLocalizer, pose_from_cluster
from hloc.utils.parsers import parse_retrieval

from simulation_pose_utils import (
    image_basename,
    load_json,
    load_pose_records,
    metadata_pose_center,
    pycolmap_transform_rt,
    quat_wxyz_to_rotmat,
)

# Dataset with query metadata.
dataset_root = Path('../datasets/simulation_26_4_26')
intrinsics_json = dataset_root / 'metadata/intrinsics_pinhole.json'

# Query image folder. Can contain any camera as long as basenames exist in poses.json.
query_source_dir = dataset_root / 'test'
query_glob = '*.png'
max_queries = 50
random_seed = 42

# Known-pose map bundle created by reconstruction_known_pose.ipynb.
map_bundle_root = Path('../outputs/simulation-known-pose-bundle')
sfm_model_root = map_bundle_root / 'sfm'
db_features = map_bundle_root / 'features.h5'
db_global_features = map_bundle_root / 'global-feats-netvlad.h5'

# Query outputs.
results_dir = map_bundle_root / 'query_batch_results_v2'
query_cache_dir = map_bundle_root / 'query_batch_v2'
results_dir.mkdir(parents=True, exist_ok=True)
query_cache_dir.mkdir(parents=True, exist_ok=True)

# Localization parameters.
num_loc = 10
max_error = 12
overwrite_query_features = False

feature_conf = copy.deepcopy(extract_features.confs['superpoint_max'])
retrieval_conf = extract_features.confs['netvlad']
matcher_conf = match_features.confs['superpoint+lightglue']

# Windows/sandbox-safe HLoc execution: avoid multiprocessing DataLoader workers.
# Keep class-compatible with Kornia's DataLoader[Any] annotation.
_original_dataloader = torch.utils.data.DataLoader
class _SingleProcessDataLoader(_original_dataloader):
    @classmethod
    def __class_getitem__(cls, item):
        return cls

    def __init__(self, *args, **kwargs):
        kwargs['num_workers'] = 0
        kwargs['pin_memory'] = False
        super().__init__(*args, **kwargs)
torch.utils.data.DataLoader = _SingleProcessDataLoader

for path in [query_source_dir, sfm_model_root, db_features, db_global_features]:
    if not path.exists():
        raise FileNotFoundError(path)

intrinsics_cfg = load_json(intrinsics_json)['cameras'][0]
print(f'Dataset: {dataset_root}')
print(f'Query dir: {query_source_dir}')
print(f'Map bundle: {map_bundle_root}')
print(f'SfM model: {sfm_model_root}')
print(f'Intrinsics: {intrinsics_cfg["model"]} {intrinsics_cfg["width"]}x{intrinsics_cfg["height"]} params={intrinsics_cfg["params"]}')

Dataset: ..\datasets\simulation_26_4_26
Query dir: ..\datasets\simulation_26_4_26\test
Map bundle: ..\outputs\simulation-known-pose-bundle
SfM model: ..\outputs\simulation-known-pose-bundle\sfm
Intrinsics: PINHOLE 1920x1080 params=[554.2562584220409, 554.2562584220409, 960.0, 540.0]


## 2. Load Map and Query Ground Truth

In [2]:
model = pycolmap.Reconstruction(sfm_model_root)
print(model.summary())

all_pose_records, pose_by_name, _ = load_pose_records(dataset_root, camera_names=None)
query_paths_all = sorted(p for p in query_source_dir.glob(query_glob) if p.is_file())
if max_queries is not None and max_queries < len(query_paths_all):
    rng = random.Random(random_seed)
    query_paths = sorted(rng.sample(query_paths_all, max_queries))
else:
    query_paths = query_paths_all

missing_metadata = [p.name for p in query_paths if p.name not in pose_by_name]
if missing_metadata:
    raise ValueError(f'{len(missing_metadata)} query images have no poses.json metadata. Examples: {missing_metadata[:10]}')

query_camera_distribution = {}
for p in query_paths:
    cam = pose_by_name[p.name]['camera_name']
    query_camera_distribution[cam] = query_camera_distribution.get(cam, 0) + 1

references = sorted(image.name for image in model.images.values())
print(f'All query images found: {len(query_paths_all)}')
print(f'Queries selected: {len(query_paths)}')
print(f'Random seed: {random_seed if max_queries is not None else None}')
print(f'Query camera distribution: {query_camera_distribution}')
print(f'Reference images in map: {len(references)}')

Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 203
	num_reg_frames = 203
	num_images = 203
	num_points3D = 89514
	num_observations = 380992
	mean_track_length = 4.25623
	mean_observations_per_image = 1876.81
	mean_reprojection_error = 1.09782
All query images found: 203
Queries selected: 100
Random seed: 31
Query camera distribution: {'back': 100}
Reference images in map: 203


## 3. Helper Functions

In [3]:
def extract_on_cpu(conf, image_root, image_list, feature_path, overwrite=True):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    original = torch.cuda.is_available
    torch.cuda.is_available = lambda: False
    try:
        return extract_features.main(
            conf,
            image_root,
            image_list=image_list,
            feature_path=feature_path,
            overwrite=overwrite,
        )
    finally:
        torch.cuda.is_available = original


def resolve_reference_ids(model, names):
    ref_ids = []
    missing = []
    for name in names:
        image = model.find_image_with_name(name)
        if image is None:
            basename = image_basename(name)
            for candidate in model.images.values():
                if image_basename(candidate.name) == basename:
                    image = candidate
                    break
        if image is None:
            missing.append(name)
        else:
            ref_ids.append(image.image_id)
    if missing:
        raise ValueError('Retrieved images not registered in model: ' + ', '.join(missing))
    return ref_ids


CARLA_TO_COLMAP_S = np.array([
    [0.0, 1.0, 0.0],
    [0.0, 0.0, -1.0],
    [1.0, 0.0, 0.0],
], dtype=np.float64)
COLMAP_TO_CARLA_S = np.linalg.inv(CARLA_TO_COLMAP_S)


def wrap_angle_deg(angle):
    return ((float(angle) + 180.0) % 360.0) - 180.0


def heading_error_deg(est_heading, gt_heading):
    return abs(wrap_angle_deg(float(est_heading) - float(gt_heading)))


def colmap_world_to_camera_to_carla_yaw_deg(r_wc_rh):
    # PnP/pycolmap returns COLMAP right-handed world-to-camera rotation.
    # Convert it back to CARLA left-handed camera-to-world rotation before extracting yaw.
    r_cw_rh = np.asarray(r_wc_rh, dtype=np.float64).T
    r_cw_lh = COLMAP_TO_CARLA_S @ r_cw_rh @ CARLA_TO_COLMAP_S
    return wrap_angle_deg(np.degrees(np.arctan2(r_cw_lh[1, 0], r_cw_lh[0, 0])))


def metadata_record_to_carla_yaw_deg(record):
    r_wc_rh = quat_wxyz_to_rotmat([record['qw'], record['qx'], record['qy'], record['qz']])
    return colmap_world_to_camera_to_carla_yaw_deg(r_wc_rh)


metadata_yaw_errors = []
for query_path in query_paths:
    rec = pose_by_name[query_path.name]
    metadata_yaw_errors.append(heading_error_deg(metadata_record_to_carla_yaw_deg(rec), rec['yaw_deg']))
metadata_yaw_errors = np.array(metadata_yaw_errors, dtype=np.float64)
print('Metadata quaternion -> CARLA yaw sanity check')
print(f'  count: {len(metadata_yaw_errors)}')
print(f'  mean error: {metadata_yaw_errors.mean():.9f} deg')
print(f'  max error: {metadata_yaw_errors.max():.9f} deg')
if metadata_yaw_errors.max() >= 1e-3:
    raise AssertionError('Metadata quaternion to CARLA yaw conversion sanity check failed.')


localizer_conf = {
    'estimation': {'ransac': {'max_error': max_error}},
    'refinement': {'refine_focal_length': False, 'refine_extra_params': False},
}
localizer = QueryLocalizer(model, localizer_conf)
camera = pycolmap.Camera(
    model=intrinsics_cfg['model'],
    width=int(intrinsics_cfg['width']),
    height=int(intrinsics_cfg['height']),
    params=np.array(intrinsics_cfg['params'], dtype=float),
)

Metadata quaternion -> CARLA yaw sanity check
  count: 100
  mean error: 0.000002423 deg
  max error: 0.000007664 deg


## 4. Batch Query Localization

In [4]:
localization_results = []

for idx, query_path in enumerate(query_paths, start=1):
    query_basename = query_path.name
    query_gt = pose_by_name[query_basename]
    query_gt_center = metadata_pose_center(query_gt)
    query_dst = query_cache_dir / query_basename
    shutil.copy2(query_path, query_dst)
    query_rel = f'{query_cache_dir.name}/{query_basename}'

    print(f'\n[{idx}/{len(query_paths)}] {query_basename} camera={query_gt["camera_name"]}')

    stem = Path(query_basename).stem
    query_features = results_dir / f'{stem}-features.h5'
    query_global_features = results_dir / f'{stem}-global-feats-netvlad.h5'
    query_matches = results_dir / f'{stem}-matches.h5'
    loc_pairs = results_dir / f'{stem}-pairs-query-netvlad.txt'

    try:
        extract_on_cpu(retrieval_conf, map_bundle_root, [query_rel], query_global_features, overwrite=overwrite_query_features)
        extract_on_cpu(feature_conf, map_bundle_root, [query_rel], query_features, overwrite=overwrite_query_features)

        pairs_from_retrieval.main(
            descriptors=query_global_features,
            output=loc_pairs,
            num_matched=min(num_loc, len(references)),
            query_list=[query_rel],
            db_list=references,
            db_descriptors=db_global_features,
        )

        match_features.main(
            matcher_conf,
            loc_pairs,
            features=query_features,
            features_ref=db_features,
            matches=query_matches,
            overwrite=True,
        )

        retrieval_dict = parse_retrieval(loc_pairs)
        retrieved_names = retrieval_dict[query_rel]
        ref_ids = resolve_reference_ids(model, retrieved_names)

        ret, log = pose_from_cluster(localizer, query_rel, camera, ref_ids, query_features, query_matches)
        if ret is None:
            print('  pose estimation failed')
            localization_results.append({
                'image_name': query_basename,
                'camera_name': query_gt['camera_name'],
                'success': False,
                'error': 'pose_from_cluster returned None',
                'retrieved': retrieved_names,
            })
            continue

        R_wc_rh, tvec = pycolmap_transform_rt(ret['cam_from_world'])
        estimated_center = -R_wc_rh.T @ tvec
        position_error_m = float(np.linalg.norm(estimated_center - query_gt_center))

        estimated_heading = colmap_world_to_camera_to_carla_yaw_deg(R_wc_rh)
        gt_heading = float(query_gt['yaw_deg'])
        heading_error = heading_error_deg(estimated_heading, gt_heading)

        result = {
            'image_name': query_basename,
            'camera_name': query_gt['camera_name'],
            'capture_id': int(query_gt['capture_id']),
            'success': True,
            'num_inliers': int(ret['num_inliers']),
            'position_error_m': position_error_m,
            'estimated_center_x': float(estimated_center[0]),
            'estimated_center_y': float(estimated_center[1]),
            'estimated_center_z': float(estimated_center[2]),
            'gt_center_x': float(query_gt_center[0]),
            'gt_center_y': float(query_gt_center[1]),
            'gt_center_z': float(query_gt_center[2]),
            'estimated_heading_deg': estimated_heading,
            'ground_truth_yaw_deg': gt_heading,
            'heading_error_deg': float(heading_error),
            'retrieved': retrieved_names,
        }
        localization_results.append(result)
        print(f"  success inliers={result['num_inliers']} pos_err={position_error_m:.3f} m heading_err={heading_error:.2f} deg")

    except Exception as exc:
        print(f'  error: {exc}')
        localization_results.append({
            'image_name': query_basename,
            'camera_name': query_gt['camera_name'],
            'capture_id': int(query_gt['capture_id']),
            'success': False,
            'error': str(exc),
        })

print('\nBatch localization finished')

[2026/04/26 13:00:52 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'netvlad'},
 'output': 'global-feats-netvlad',
 'preprocessing': {'resize_max': 1024}}
[2026/04/26 13:00:52 hloc INFO] Skipping the extraction.
[2026/04/26 13:00:52 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2026/04/26 13:00:52 hloc INFO] Skipping the extraction.
[2026/04/26 13:00:52 hloc INFO] Extracting image pairs from a retrieval database.



[1/100] 000004_back_f00018784.png camera=back


[2026/04/26 13:00:52 hloc INFO] Found 5 pairs.
[2026/04/26 13:00:52 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
c:\Users\ilker\Desktop\bitirme-project\venv38\lib\site-packages\kornia\feature\lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
  0%|          | 0/5 [00:00<?, ?it/s]c:\Users\ilker\Desktop\bitirme-project\venv38\lib\site-packages\lightglue\lightglue.py:120: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  v = F.scaled_dot_product_attention(*args, attn_mask=mask).to(q.dtype)
100%|██████████| 5/5 [00:00<00:00,  9.12it/s]
[2026/04/26 13:00:54 hloc INFO

  success inliers=127 pos_err=0.092 m heading_err=0.15 deg

[2/100] 000006_back_f00018852.png camera=back


100%|██████████| 1/1 [00:01<00:00,  1.11s/it]
[2026/04/26 13:01:00 hloc INFO] Finished exporting features.
[2026/04/26 13:01:00 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}


...

  success inliers=487 pos_err=0.004 m heading_err=0.01 deg

[100/100] 000203_back_f00028800.png camera=back


100%|██████████| 1/1 [00:01<00:00,  1.42s/it]
[2026/04/26 13:12:59 hloc INFO] Finished exporting features.
[2026/04/26 13:12:59 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}


Loaded SuperPoint model


100%|██████████| 1/1 [00:03<00:00,  3.47s/it]
[2026/04/26 13:13:03 hloc INFO] Finished exporting features.
[2026/04/26 13:13:03 hloc INFO] Extracting image pairs from a retrieval database.
[2026/04/26 13:13:03 hloc INFO] Found 5 pairs.
[2026/04/26 13:13:03 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
100%|██████████| 5/5 [00:00<00:00,  5.54it/s]
[2026/04/26 13:13:04 hloc INFO] Finished exporting matches.


  success inliers=711 pos_err=0.007 m heading_err=0.02 deg

Batch localization finished


## 5. Results Summary

In [5]:
results_df = pd.DataFrame(localization_results)
display(results_df)

successful = results_df[results_df['success'] == True]
failed = results_df[results_df['success'] != True]

summary = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'dataset_root': str(dataset_root),
    'query_source_dir': str(query_source_dir),
    'map_bundle_root': str(map_bundle_root),
    'sfm_model_root': str(sfm_model_root),
    'total_queries': int(len(results_df)),
    'successful_queries': int(len(successful)),
    'failed_queries': int(len(failed)),
    'success_rate': float(len(successful) / max(1, len(results_df))),
    'num_retrieved': num_loc,
    'ransac_max_error_px': max_error,
}

if len(successful) > 0:
    summary['position_error_m'] = {
        'mean': float(successful['position_error_m'].mean()),
        'median': float(successful['position_error_m'].median()),
        'max': float(successful['position_error_m'].max()),
    }
    summary['inliers'] = {
        'mean': float(successful['num_inliers'].mean()),
        'median': float(successful['num_inliers'].median()),
        'min': int(successful['num_inliers'].min()),
    }
    if 'heading_error_deg' in successful:
        summary['heading_error_deg'] = {
            'mean': float(successful['heading_error_deg'].mean()),
            'median': float(successful['heading_error_deg'].median()),
            'max': float(successful['heading_error_deg'].max()),
        }

print(json.dumps(summary, indent=2))

details_csv = results_dir / 'query_batch_details.csv'
summary_json = results_dir / 'query_batch_summary.json'
results_df.to_csv(details_csv, index=False)
summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Details CSV: {details_csv}')
print(f'Summary JSON: {summary_json}')

,image_name,camera_name,capture_id,success,num_inliers,position_error_m,estimated_center_x,estimated_center_y,estimated_center_z,gt_center_x,gt_center_y,gt_center_z,estimated_heading_deg,ground_truth_yaw_deg,heading_error_deg,retrieved
0,000004_back_f00018784.png,back,4,True,127,0.092479,2.684087,-2.267377,9.355537,2.656292,-2.201233,9.297187,-157.001757,-156.856216,0.145540,"[images/000202_front_f00028773.png, images/000..."
1,000006_back_f00018852.png,back,6,True,183,0.022101,3.044670,-2.221512,10.183425,3.036942,-2.202065,10.190533,-156.781694,-156.815170,0.033476,"[images/000202_front_f00028773.png, images/000..."
2,000008_back_f00018932.png,back,8,True,259,0.058042,3.470729,-2.246350,11.101116,3.436513,-2.202409,11.117469,-156.879789,-156.929489,0.049700,"[images/000202_front_f00028773.png, images/000..."
3,000009_back_f00019027.png,back,9,True,269,0.126330,3.655781,-2.222138,11.952341,3.768445,-2.201920,11.898887,-157.035918,-156.899445,0.136474,"[images/000202_front_f00028773.png, images/000..."
4,000011_back_f00019114.png,back,11,True,392,0.048445,4.178143,-2.226588,12.784634,4.163083,-2.201742,12.823400,-156.842878,-156.882858,0.039981,"[images/000202_front_f00028773.png, images/000..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,000198_back_f00028564.png,back,198,True,509,0.003898,11.602025,-2.209379,24.605108,11.602754,-2.206892,24.608019,23.802462,23.820953,0.018491,"[images/000025_front_f00019995.png, images/000..."
96,000199_back_f00028624.png,back,199,True,484,0.003182,10.742547,-2.205126,22.660010,10.744089,-2.204666,22.662755,23.866332,23.887154,0.020821,"[images/000025_front_f00019995.png, images/000..."
97,000200_back_f00028669.png,back,200,True,454,0.014977,10.171062,-2.200921,21.331684,10.156401,-2.203795,21.332724,23.942694,23.978243,0.035549,"[images/000020_front_f00019752.png, images/000..."
98,000201_back_f00028713.png,back,201,True,487,0.004008,9.611357,-2.203877,20.114541,9.612705,-2.204114,20.110775,23.989179,23.977945,0.011234,"[images/000019_front_f00019679.png, images/000..."


{
  "timestamp": "2026-04-26T13:13:05",
  "dataset_root": "..\\datasets\\simulation_26_4_26",
  "query_source_dir": "..\\datasets\\simulation_26_4_26\\test",
  "map_bundle_root": "..\\outputs\\simulation-known-pose-bundle",
  "sfm_model_root": "..\\outputs\\simulation-known-pose-bundle\\sfm",
  "total_queries": 100,
  "successful_queries": 100,
  "failed_queries": 0,
  "success_rate": 1.0,
  "num_retrieved": 5,
  "ransac_max_error_px": 3,
  "position_error_m": {
    "mean": 0.17796639741561399,
    "median": 0.009015521976386161,
    "max": 8.00238724187634
  },
  "inliers": {
    "mean": 989.02,
    "median": 830.5,
    "min": 127
  },
  "heading_error_deg": {
    "mean": 0.019412697842079468,
    "median": 0.010392953271662009,
    "max": 0.14554015363805206
  }
}
Details CSV: ..\outputs\simulation-known-pose-bundle\query_batch_results_v2\query_batch_details.csv
Summary JSON: ..\outputs\simulation-known-pose-bundle\query_batch_results_v2\query_batch_summary.json


## 6. VisualLocalization.net Threshold Format

In [6]:
benchmark_thresholds = [
    (0.25, 2.0),
    (0.50, 5.0),
    (5.00, 10.0),
]

total_queries = len(results_df)
successful_eval = successful.dropna(subset=['position_error_m', 'heading_error_deg']).copy()

benchmark_parts = []
benchmark_rows = []
for pos_thr, rot_thr in benchmark_thresholds:
    passed = successful_eval[
        (successful_eval['position_error_m'] <= pos_thr)
        & (successful_eval['heading_error_deg'] <= rot_thr)
    ]
    count = int(len(passed))
    percent_total = 100.0 * count / max(1, total_queries)
    percent_successful = 100.0 * count / max(1, len(successful_eval))
    benchmark_parts.append(f'{percent_total:.1f}')
    benchmark_rows.append({
        'position_threshold_m': pos_thr,
        'heading_threshold_deg': rot_thr,
        'passed': count,
        'total_queries': int(total_queries),
        'successful_evaluated_queries': int(len(successful_eval)),
        'percent_of_all_queries': percent_total,
        'percent_of_successful_queries': percent_successful,
    })

benchmark_df = pd.DataFrame(benchmark_rows)
print('VisualLocalization.net style localization scores')
print('All conditions: (0.25m, 2 deg) / (0.5m, 5 deg) / (5m, 10 deg)')
print('All conditions: ' + ' / '.join(benchmark_parts))
display(benchmark_df)

benchmark_json = results_dir / 'query_batch_visual_localization_thresholds.json'
benchmark_json.write_text(json.dumps(benchmark_rows, indent=2), encoding='utf-8')
print(f'Benchmark thresholds JSON: {benchmark_json}')

VisualLocalization.net style localization scores
All conditions: (0.25m, 2 deg) / (0.5m, 5 deg) / (5m, 10 deg)
All conditions: 98.0 / 98.0 / 98.0


,position_threshold_m,heading_threshold_deg,passed,total_queries,successful_evaluated_queries,percent_of_all_queries,percent_of_successful_queries
0,0.25,2.0,98,100,100,98.0,98.0
1,0.50,5.0,98,100,100,98.0,98.0
2,5.00,10.0,98,100,100,98.0,98.0


Benchmark thresholds JSON: ..\outputs\simulation-known-pose-bundle\query_batch_results_v2\query_batch_visual_localization_thresholds.json


## 7. Worst Successful Queries

In [7]:
if len(successful) > 0:
    worst = successful.sort_values('position_error_m', ascending=False).head(10)
    display(worst[['image_name', 'camera_name', 'capture_id', 'num_inliers', 'position_error_m', 'heading_error_deg']])
else:
    print('No successful queries to inspect.')

if len(failed) > 0:
    print('Failed queries:')
    display(failed[['image_name', 'camera_name', 'capture_id', 'error']])

,image_name,camera_name,capture_id,num_inliers,position_error_m,heading_error_deg
59,000113_back_f00024230.png,back,113,623,8.002387,0.025000
56,000107_back_f00023984.png,back,107,1178,8.002208,0.032540
3,000009_back_f00019027.png,back,9,269,0.126330,0.136474
11,000024_back_f00019951.png,back,24,850,0.105526,0.094832
0,000004_back_f00018784.png,back,4,127,0.092479,0.145540
15,000031_back_f00020223.png,back,31,555,0.080975,0.012106
13,000026_back_f00020054.png,back,26,704,0.069645,0.051022
12,000025_back_f00019995.png,back,25,739,0.068785,0.044087
2,000008_back_f00018932.png,back,8,259,0.058042,0.049700
9,000016_back_f00019487.png,back,16,604,0.056975,0.081965
